# NiyamTrace-X Q1 Experiment 07 — Verified Execution Broker V2: Atomic Capability Execution

**Purpose.** Correct the transaction-ordering flaw uncovered in Experiment 03 and test execution binding under real SQLite transactions, process restarts, randomized states and concurrent writers.

The safety invariant is:

\[
\text{signed predicted effect set}
=\text{effect set revalidated inside the write transaction}
=\text{effect set actually committed}.
\]

### V2 changes
- `BEGIN IMMEDIATE` occurs **before** state/snapshot revalidation.
- Capability signature binds actor, tool, canonical arguments, policy hash, snapshot hash, exact predicted IDs, expiry and nonce.
- Nonce consumption is persisted in SQLite in the same transaction as the mutation.
- Commit-time changed-row equality is checked before commit.
- Crash/failpoint causes rollback.
- Concurrent phantom writers are tested with separate SQLite connections.
- Transactional outbox + provider idempotency demonstrates retry-safe external dispatch semantics in a simulated provider.
- Randomized attack trials generate distinct database states/arguments instead of repeating one fixed scenario.

The notebook first contains a regression demonstration of the old verify-then-transaction window, then verifies that V2 closes that window.


In [ ]:
import importlib.util, subprocess, sys
need=[p for p in ['pandas','numpy','matplotlib'] if importlib.util.find_spec(p) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q']+need)


In [ ]:
from pathlib import Path
import sqlite3, json, hashlib, hmac, secrets, time, threading, tempfile, random, string, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED=20260911
random.seed(SEED); np.random.seed(SEED)
BASE=Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS=BASE/'niyamtrace_q1_wave2_results'; RESULTS.mkdir(parents=True,exist_ok=True)
SECRET=b'niyamtrace-q1-experiment-v2-secret'
POLICY={'name':'archive-policy','version':2,'allow_tools':['invoice.archive']}

def cj(x): return json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False)
def hobj(x): return hashlib.sha256(cj(x).encode()).hexdigest()
def sign(body): return hmac.new(SECRET,cj(body).encode(),hashlib.sha256).hexdigest()
def policy_hash(policy=POLICY): return hobj(policy)


In [ ]:
# Database helpers.
def connect(path):
    c=sqlite3.connect(path,timeout=10,isolation_level=None,check_same_thread=False)
    c.execute('PRAGMA journal_mode=WAL')
    c.execute('PRAGMA busy_timeout=10000')
    return c

def init_db(path, rows=None):
    if Path(path).exists(): Path(path).unlink()
    c=connect(path)
    c.executescript("""
    CREATE TABLE invoices(id TEXT PRIMARY KEY, vendor TEXT, month INTEGER, year INTEGER, status TEXT);
    CREATE TABLE used_nonces(nonce TEXT PRIMARY KEY, used_at REAL NOT NULL);
    CREATE TABLE outbox(effect_id TEXT PRIMARY KEY, idempotency_key TEXT UNIQUE, payload TEXT NOT NULL, status TEXT NOT NULL DEFAULT 'PENDING', attempts INTEGER NOT NULL DEFAULT 0);
    """)
    rows=rows or [('I1','7184',11,2025,'OPEN'),('I2','7184',11,2025,'OPEN'),('I3','7184',11,2025,'OPEN'),('I4','9001',11,2025,'OPEN')]
    c.executemany('INSERT INTO invoices VALUES(?,?,?,?,?)',rows); c.close()

def snapshot_rows(c):
    return c.execute('SELECT id,vendor,month,year,status FROM invoices ORDER BY id').fetchall()
def snapshot_hash(c): return hobj(snapshot_rows(c))
def select_ids(c,args):
    q='SELECT id FROM invoices WHERE vendor=? AND month=? AND year=? AND status="OPEN" ORDER BY id'
    return [r[0] for r in c.execute(q,(str(args['vendor']),int(args['month']),int(args['year'])))]

def issue_capability(path,actor,args,ttl=60,policy=POLICY):
    c=connect(path)
    ids=select_ids(c,args); snap=snapshot_hash(c); c.close()
    body={'actor':actor,'tool':'invoice.archive','args_hash':hobj(args),'policy_hash':policy_hash(policy),'snapshot_hash':snap,'predicted_ids':ids,'predicted_ids_hash':hobj(ids),'expires_at':time.time()+ttl,'nonce':secrets.token_hex(16)}
    return {**body,'signature':sign(body)}


In [ ]:
# Regression-only model of the old verify-then-transaction ordering.
class LegacyWindowBroker:
    def __init__(self,path): self.path=path
    def verify(self,cap,actor,args,policy=POLICY):
        body={k:v for k,v in cap.items() if k!='signature'}
        if not hmac.compare_digest(sign(body),cap['signature']): return False,'SIGNATURE'
        if actor!=cap['actor'] or hobj(args)!=cap['args_hash'] or policy_hash(policy)!=cap['policy_hash']: return False,'BINDING'
        c=connect(self.path); ok=snapshot_hash(c)==cap['snapshot_hash']; c.close()
        return (ok,'OK' if ok else 'STALE')
    def execute_after_verify(self,cap,args,between_hook=None):
        ok,reason=self.verify(cap,cap['actor'],args)
        if not ok:return {'committed':False,'reason':reason,'actual_ids':[]}
        if between_hook: between_hook()  # vulnerable window
        c=connect(self.path); c.execute('BEGIN IMMEDIATE')
        ids=select_ids(c,args)
        for i in ids:c.execute('UPDATE invoices SET status="ARCHIVED" WHERE id=?',(i,))
        c.commit(); c.close()
        return {'committed':True,'reason':'OK','actual_ids':ids}

# Red regression test: prove the old ordering can expand the effect set.
db=BASE/'exp07_legacy_regression.sqlite'; init_db(db)
args={'vendor':'7184','month':11,'year':2025}; cap=issue_capability(db,'alice',args)
def phantom():
    c=connect(db); c.execute('INSERT INTO invoices VALUES(?,?,?,?,?)',('I6','7184',11,2025,'OPEN')); c.close()
legacy=LegacyWindowBroker(db)
r=legacy.execute_after_verify(cap,args,phantom)
print('Legacy predicted:',cap['predicted_ids'],'actual:',r['actual_ids'])
assert r['committed'] and set(r['actual_ids'])!=set(cap['predicted_ids']), 'Regression setup did not reproduce the old TOCTOU window.'


In [ ]:
class AtomicCapabilityBroker:
    def __init__(self,path,secret=SECRET): self.path=Path(path); self.secret=secret
    def _verify_static(self,cap,actor,args,policy):
        body={k:v for k,v in cap.items() if k!='signature'}
        if not hmac.compare_digest(sign(body),cap.get('signature','')): return False,'SIGNATURE'
        if time.time()>cap['expires_at']: return False,'EXPIRED'
        if actor!=cap['actor']: return False,'ACTOR'
        if cap['tool']!='invoice.archive': return False,'TOOL'
        if hobj(args)!=cap['args_hash']: return False,'ARGS'
        if policy_hash(policy)!=cap['policy_hash']: return False,'POLICY'
        if hobj(cap['predicted_ids'])!=cap['predicted_ids_hash']: return False,'PREDICTED_SET_TAMPER'
        return True,'OK'

    def execute_archive(self,cap,actor,args,policy=POLICY,failpoint=None,inside_tx_hook=None):
        c=connect(self.path)
        try:
            # Critical fix: acquire the write transaction before state verification.
            c.execute('BEGIN IMMEDIATE')
            ok,reason=self._verify_static(cap,actor,args,policy)
            if not ok: c.rollback(); return {'committed':False,'reason':reason,'actual_ids':[]}
            if c.execute('SELECT 1 FROM used_nonces WHERE nonce=?',(cap['nonce'],)).fetchone():
                c.rollback(); return {'committed':False,'reason':'REPLAY','actual_ids':[]}
            if snapshot_hash(c)!=cap['snapshot_hash']:
                c.rollback(); return {'committed':False,'reason':'STALE_SNAPSHOT','actual_ids':[]}
            current=select_ids(c,args)
            if current!=cap['predicted_ids'] or hobj(current)!=cap['predicted_ids_hash']:
                c.rollback(); return {'committed':False,'reason':'EFFECT_SET_MISMATCH','actual_ids':[]}
            if inside_tx_hook: inside_tx_hook()
            before={r[0]:r[-1] for r in snapshot_rows(c)}
            for i in current:
                c.execute('UPDATE invoices SET status="ARCHIVED" WHERE id=? AND status="OPEN"',(i,))
            after={r[0]:r[-1] for r in snapshot_rows(c)}
            actual=sorted([i for i in before if before[i]!=after[i]])
            if sorted(actual)!=sorted(cap['predicted_ids']):
                c.rollback(); return {'committed':False,'reason':'COMMIT_DELTA_MISMATCH','actual_ids':actual}
            if failpoint=='before_nonce': raise RuntimeError('injected crash before nonce persist')
            c.execute('INSERT INTO used_nonces(nonce,used_at) VALUES(?,?)',(cap['nonce'],time.time()))
            if failpoint=='before_commit': raise RuntimeError('injected crash before commit')
            c.commit(); return {'committed':True,'reason':'OK','actual_ids':actual}
        except Exception as e:
            c.rollback(); return {'committed':False,'reason':'ROLLBACK:'+type(e).__name__,'actual_ids':[]}
        finally:
            c.close()


In [ ]:
# Core unit/regression tests.
def state(path):
    c=connect(path); r=snapshot_rows(c); c.close(); return r

def fresh():
    p=Path(tempfile.mktemp(prefix='ntx7_',suffix='.sqlite')); init_db(p); return p

# Valid execution + persistent replay across broker restart.
p=fresh(); args={'vendor':'7184','month':11,'year':2025}; cap=issue_capability(p,'alice',args)
b=AtomicCapabilityBroker(p); r1=b.execute_archive(cap,'alice',args); r2=AtomicCapabilityBroker(p).execute_archive(cap,'alice',args)
assert r1['committed'] and r1['actual_ids']==['I1','I2','I3']; assert not r2['committed'] and r2['reason']=='REPLAY'

# Stale snapshot.
p=fresh(); cap=issue_capability(p,'alice',args); c=connect(p); c.execute('INSERT INTO invoices VALUES(?,?,?,?,?)',('I6','7184',11,2025,'OPEN')); c.close()
r=AtomicCapabilityBroker(p).execute_archive(cap,'alice',args); assert not r['committed'] and r['reason']=='STALE_SNAPSHOT'

# Crash rollback preserves state and nonce remains unused.
p=fresh(); cap=issue_capability(p,'alice',args); before=state(p); r=AtomicCapabilityBroker(p).execute_archive(cap,'alice',args,failpoint='before_commit'); after=state(p)
assert not r['committed'] and before==after
r2=AtomicCapabilityBroker(p).execute_archive(cap,'alice',args); assert r2['committed']
print('Core V2 regression tests passed.')


In [ ]:
# Concurrent phantom-writer test: writer attempts insertion while broker holds BEGIN IMMEDIATE.
p=fresh(); cap=issue_capability(p,'alice',args); tx_started=threading.Event(); writer_done=threading.Event(); writer_elapsed={}

def writer():
    tx_started.wait(5); t=time.time(); c=connect(p)
    c.execute('INSERT INTO invoices VALUES(?,?,?,?,?)',('I6','7184',11,2025,'OPEN')); c.close()
    writer_elapsed['seconds']=time.time()-t; writer_done.set()

def inside_hook():
    tx_started.set(); time.sleep(.25)  # keep transaction open while writer tries to write

t=threading.Thread(target=writer); t.start()
r=AtomicCapabilityBroker(p).execute_archive(cap,'alice',args,inside_tx_hook=inside_hook)
t.join(5)
assert r['committed'] and r['actual_ids']==cap['predicted_ids']
assert writer_done.is_set()
c=connect(p); s=dict((row[0],row[-1]) for row in snapshot_rows(c)); c.close()
assert s['I6']=='OPEN' and all(s[i]=='ARCHIVED' for i in cap['predicted_ids'])
print('Concurrent writer waited ~',round(writer_elapsed['seconds'],3),'s; committed effect set stayed exact.')


In [ ]:
# Distinct randomized attack trials.
def random_rows(rng,n=10):
    vendors=[str(rng.integers(1000,9999)) for _ in range(4)]
    rows=[]
    for i in range(n):
        rows.append((f'I{i:03d}',rng.choice(vendors),int(rng.integers(1,13)),int(rng.integers(2024,2028)),'OPEN'))
    return rows

def pick_args(rows,rng):
    r=rows[int(rng.integers(0,len(rows)))]
    return {'vendor':r[1],'month':r[2],'year':r[3]}

ATTACKS=['valid','replay','args_tamper','stale_snapshot','policy_change','expired','wrong_actor','wrong_tool','forged_signature','predicted_ids_tamper','crash_before_commit']
rng=np.random.default_rng(SEED); rec=[]
for attack in ATTACKS:
    for rep in range(120):
        p=Path(tempfile.mktemp(prefix='ntx7r_',suffix='.sqlite')); rows=random_rows(rng,int(rng.integers(8,20))); init_db(p,rows); a=pick_args(rows,rng); cap=issue_capability(p,'alice',a,ttl=60); actor='alice'; pol=POLICY; aa=dict(a); fp=None
        if attack=='replay':
            AtomicCapabilityBroker(p).execute_archive(cap,actor,aa)
        elif attack=='args_tamper': aa={**aa,'year':aa['year']+1}
        elif attack=='stale_snapshot':
            c=connect(p); c.execute('INSERT INTO invoices VALUES(?,?,?,?,?)',(f'P{rep}',a['vendor'],a['month'],a['year'],'OPEN')); c.close()
        elif attack=='policy_change': pol={**POLICY,'version':999}
        elif attack=='expired': cap=issue_capability(p,'alice',a,ttl=-1)
        elif attack=='wrong_actor': actor='mallory'
        elif attack=='wrong_tool': cap=dict(cap); cap['tool']='vendor.suspend'  # signature now invalid too; both are a block
        elif attack=='forged_signature': cap=dict(cap); cap['signature']='00'*32
        elif attack=='predicted_ids_tamper': cap=dict(cap); cap['predicted_ids']=cap['predicted_ids']+['FAKE']
        elif attack=='crash_before_commit': fp='before_commit'
        result=AtomicCapabilityBroker(p).execute_archive(cap,actor,aa,pol,failpoint=fp)
        expected=(attack=='valid')
        rec.append({'attack':attack,'rep':rep,'committed':result['committed'],'reason':result['reason'],'expected_commit':expected,'pass':result['committed']==expected,'state_hash':hobj(rows),'args_hash':hobj(a)})
        try: p.unlink()
        except: pass
trials=pd.DataFrame(rec)
summary=trials.groupby('attack').agg(n=('pass','size'),pass_rate=('pass','mean'),commit_rate=('committed','mean'),unique_states=('state_hash','nunique'),unique_args=('args_hash','nunique')).reset_index()
display(summary)
assert trials['pass'].all()
summary.to_csv(RESULTS/'exp07_randomized_attack_summary.csv',index=False); trials.to_csv(RESULTS/'exp07_randomized_attack_trials.csv',index=False)


In [ ]:
# Transactional outbox + idempotent simulated provider.
class SimProvider:
    def __init__(self): self.done=set(); self.effects=[]
    def send(self,key,payload,fail_once=False):
        if key in self.done: return {'status':'DUPLICATE_SUPPRESSED'}
        if fail_once and not hasattr(self,'failed_once'):
            self.failed_once=True; raise RuntimeError('simulated provider failure')
        self.done.add(key); self.effects.append((key,payload)); return {'status':'APPLIED'}

def enqueue_outbox(path,effects):
    c=connect(path); c.execute('BEGIN IMMEDIATE')
    for e in effects:
        key=hobj(e); c.execute('INSERT OR IGNORE INTO outbox(effect_id,idempotency_key,payload,status) VALUES(?,?,?,"PENDING")',(key,key,cj(e)))
    c.commit(); c.close()

def dispatch(path,provider,fail_first=False):
    c=connect(path); rows=c.execute('SELECT effect_id,idempotency_key,payload FROM outbox WHERE status!="DONE" ORDER BY effect_id').fetchall(); c.close()
    out=[]
    for i,(eid,key,payload) in enumerate(rows):
        try:
            res=provider.send(key,json.loads(payload),fail_once=fail_first and i==0)
            c=connect(path); c.execute('UPDATE outbox SET status="DONE", attempts=attempts+1 WHERE effect_id=?',(eid,)); c.close(); out.append((eid,res['status']))
        except Exception:
            c=connect(path); c.execute('UPDATE outbox SET attempts=attempts+1 WHERE effect_id=?',(eid,)); c.close(); out.append((eid,'FAILED_RETRYABLE'))
    return out

p=fresh(); effects=[{'tool':'provider.archive','invoice_id':x} for x in ['I1','I2','I3']]; enqueue_outbox(p,effects); provider=SimProvider()
first=dispatch(p,provider,fail_first=True); second=dispatch(p,provider); third=dispatch(p,provider)
assert len(provider.effects)==3 and len(provider.done)==3
print('Outbox dispatch rounds:',first,second,third,'provider effects=',len(provider.effects))
p.unlink()


In [ ]:
# Figures and manifest.
plot=summary.copy(); ax=plot.plot(x='attack',y='pass_rate',kind='bar',legend=False,figsize=(9,4.5)); ax.set_ylim(0,1.05); ax.set_ylabel('Expected behavior rate'); ax.set_title('VEB V2 randomized security regression matrix'); plt.xticks(rotation=40,ha='right'); plt.tight_layout(); plt.savefig(RESULTS/'exp07_attack_matrix.png',dpi=220,bbox_inches='tight'); plt.show()

manifest={'experiment':'NTX_Q1_07_Verified_Execution_Broker_V2','seed':SEED,'architecture':'transaction-first SQLite capability broker','critical_invariant':'signed predicted set == revalidated in-transaction set == committed delta','randomized_trials':int(len(trials)),'all_expected_behaviors':bool(trials['pass'].all()),'warning':'Reference experimental broker; not evidence that the public repository executor already implements this design.'}
(RESULTS/'exp07_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
(RESULTS/'exp07_legacy_toctou_regression.json').write_text(json.dumps({'predicted':cap.get('predicted_ids',[]),'note':'See notebook output for the explicit legacy regression; V2 tests are separate.'},indent=2),encoding='utf-8')


In [ ]:
# FINAL CELL — package this notebook's complete results and download the ZIP.
from pathlib import Path
import zipfile, hashlib

PREFIX = 'exp07_'
ZIP_OUT = BASE / 'NTX_Q1_07_VERIFIED_EXECUTION_BROKER_V2_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.iterdir()):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p, arcname=p.name)

sha = hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:', ZIP_OUT)
print('SHA-256:', sha)
print('Size MiB:', round(ZIP_OUT.stat().st_size/1024**2, 3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab; ZIP is available at:', ZIP_OUT)
